In [2]:
# Run configuration: edit only these three values for each run
MODEL    = 'full'        # 'vanilla' | 'baseline' | 'ordinal' | 'attention' | 'full'
BACKBONE = 'resnet18'    # 'resnet18' | 'densenet121'
SEED     = 0             # 0 | 1 | 2

import os
import json
import time
import random
import zipfile
import platform

import numpy as np
import torch
import torchvision

assert MODEL in ['vanilla', 'baseline', 'ordinal', 'attention', 'full']
assert BACKBONE in ['resnet18', 'densenet121']
assert SEED in [0, 1, 2]

RUN_NAME = f"{MODEL}_{BACKBONE}_seed{SEED}"

# Hyperparameters shared by all runs
CFG = dict(
    model=MODEL,
    backbone=BACKBONE,
    seed=SEED,
    run_name=RUN_NAME,
    img_size=224,
    epochs=30,
    lr=1e-4,
    lr_factor=0.5,
    lr_patience=5,
    n_support=5,             # per class, per episode
    n_query=5,               # per class, per episode
    episodes_per_epoch=115,  # 115 episodes x 50 images ~ 5,778 training images
    batch_size=32,           # vanilla CNN training
    ordinal_alpha=0.5,
    ordinal_margin=0.5,
    cbam_reduction=16,
)
CFG['use_ordinal'] = MODEL in ['ordinal', 'full']
CFG['use_attention'] = MODEL in ['attention', 'full']


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type != 'cuda':
    raise RuntimeError("GPU not available. Select Runtime > Change runtime type > T4 GPU.")
print(f"Device: {torch.cuda.get_device_name(0)}")

from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/few_short'
ZIP_PATH = os.path.join(PROJECT, 'data.zip')
RUN_DIR = os.path.join(PROJECT, 'runs', RUN_NAME)
if not os.path.exists(ZIP_PATH):
    raise FileNotFoundError(ZIP_PATH)
os.makedirs(RUN_DIR, exist_ok=True)

if os.path.exists(os.path.join(RUN_DIR, 'results.json')):
    print(f"Note: {RUN_NAME} already has results.json. Check the configuration before training.")

# Copy the dataset to the local disk; reading thousands of files from Drive is slow
LOCAL_ROOT = '/content/knee_data'
if not os.path.isdir(LOCAL_ROOT):
    t0 = time.time()
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(LOCAL_ROOT)
    print(f"Unzipped data in {time.time() - t0:.0f} s")

DATA_DIR = None
for root, dirs, _ in os.walk(LOCAL_ROOT):
    if {'train', 'val', 'test'} <= set(dirs):
        DATA_DIR = root
        break
if DATA_DIR is None:
    raise FileNotFoundError("train/val/test folders not found in data.zip")

# Expected image counts per KL grade (0-4) in the predefined patient-level split
IMG_EXT = ('.png', '.jpg', '.jpeg')
EXPECTED = {
    'train': [2286, 1046, 1516, 757, 173],
    'val':   [328, 153, 212, 106, 27],
    'test':  [639, 296, 447, 223, 51],
}
for split, expected in EXPECTED.items():
    counts = [
        len([f for f in os.listdir(os.path.join(DATA_DIR, split, str(k)))
             if f.lower().endswith(IMG_EXT)])
        for k in range(5)
    ]
    if counts != expected:
        raise ValueError(f"{split}: found {counts}, expected {expected}")
    print(f"{split:5s} {counts} total={sum(counts)}")

CFG['versions'] = dict(
    python=platform.python_version(),
    torch=torch.__version__,
    torchvision=torchvision.__version__,
    gpu=torch.cuda.get_device_name(0),
)
with open(os.path.join(RUN_DIR, 'config.json'), 'w') as f:
    json.dump(CFG, f, indent=2)

print(f"\nRun: {RUN_NAME}")
print(json.dumps(CFG, indent=2))
print(f"Data: {DATA_DIR}")
print(f"Output: {RUN_DIR}")

Device: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
train [2286, 1046, 1516, 757, 173] total=5778
val   [328, 153, 212, 106, 27] total=826
test  [639, 296, 447, 223, 51] total=1656

Run: full_resnet18_seed0
{
  "model": "full",
  "backbone": "resnet18",
  "seed": 0,
  "run_name": "full_resnet18_seed0",
  "img_size": 224,
  "epochs": 30,
  "lr": 0.0001,
  "lr_factor": 0.5,
  "lr_patience": 5,
  "n_support": 5,
  "n_query": 5,
  "episodes_per_epoch": 115,
  "batch_size": 32,
  "ordinal_alpha": 0.5,
  "ordinal_margin": 0.5,
  "cbam_reduction": 16,
  "use_ordinal": true,
  "use_attention": true,
  "versions": {
    "python": "3.13.15",
    "torch": "2.11.0+cu128",
    "torchvision": "0.26.0+cu128",
    "gpu": "Tesla T4"
  }
}
Data: /content/knee_data/data
Output: /content/drive/MyDrive/few_short/runs/full_resnet18_seed0


In [4]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader, Sampler
import torchvision.transforms as T

MEAN = [0.485, 0.456, 0.406]  # ImageNet statistics
STD = [0.229, 0.224, 0.225]

train_tf = T.Compose([
    T.Resize((CFG['img_size'], CFG['img_size'])),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=10),
    T.ColorJitter(brightness=0.2, contrast=0.2),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])
eval_tf = T.Compose([
    T.Resize((CFG['img_size'], CFG['img_size'])),
    T.ToTensor(),
    T.Normalize(MEAN, STD),
])


class KneeDataset(Dataset):
    """Images stored as DATA_DIR/<split>/<KL grade>/<file>. Files are sorted for a fixed order."""

    def __init__(self, split, transform):
        self.transform = transform
        self.paths, labels = [], []
        for k in range(5):
            class_dir = os.path.join(DATA_DIR, split, str(k))
            for fname in sorted(os.listdir(class_dir)):
                if fname.lower().endswith(IMG_EXT):
                    self.paths.append(os.path.join(class_dir, fname))
                    labels.append(k)
        self.labels = np.array(labels)
        self.files = [os.path.relpath(p, DATA_DIR).replace('\\', '/') for p in self.paths]

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert('RGB')
        return self.transform(img), int(self.labels[i])


class EpisodeSampler(Sampler):
    """
    Class-balanced episodes for prototypical training.

    Each episode draws n_support + n_query distinct images from every KL grade.
    Index layout: [support for grades 0..4] + [query for grades 0..4].
    """

    def __init__(self, labels, n_support, n_query, n_episodes, seed):
        self.class_idx = [np.where(labels == k)[0] for k in range(5)]
        self.n_support = n_support
        self.n_query = n_query
        self.n_episodes = n_episodes
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return self.n_episodes

    def __iter__(self):
        for _ in range(self.n_episodes):
            support, query = [], []
            for idx in self.class_idx:
                pick = self.rng.choice(idx, self.n_support + self.n_query, replace=False)
                support.extend(pick[:self.n_support].tolist())
                query.extend(pick[self.n_support:].tolist())
            yield support + query


def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)


loader_gen = torch.Generator()
loader_gen.manual_seed(SEED)

train_ds = KneeDataset('train', train_tf)
train_eval_ds = KneeDataset('train', eval_tf)  # no augmentation; used to build class prototypes
val_ds = KneeDataset('val', eval_tf)
test_ds = KneeDataset('test', eval_tf)

loader_kwargs = dict(num_workers=2, pin_memory=True,
                     worker_init_fn=seed_worker, generator=loader_gen)

if MODEL == 'vanilla':
    train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True, **loader_kwargs)
else:
    train_sampler = EpisodeSampler(train_ds.labels, CFG['n_support'], CFG['n_query'],
                                   CFG['episodes_per_epoch'], seed=SEED)
    train_loader = DataLoader(train_ds, batch_sampler=train_sampler, **loader_kwargs)


def make_eval_loader(ds):
    return DataLoader(ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)


train_eval_loader = make_eval_loader(train_eval_ds)
val_loader = make_eval_loader(val_ds)
test_loader = make_eval_loader(test_ds)

print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")
print(f"Training batches per epoch: {len(train_loader)}")

if MODEL != 'vanilla':
    # Inspect one episode using a separate sampler so the training sampler is untouched
    episode = next(iter(EpisodeSampler(train_ds.labels, CFG['n_support'], CFG['n_query'], 1, seed=999)))
    n_sup = 5 * CFG['n_support']
    print(f"Episode size: {len(episode)}")
    print(f"Support labels: {train_ds.labels[episode[:n_sup]].tolist()}")
    print(f"Query labels:   {train_ds.labels[episode[n_sup:]].tolist()}")
    print(f"Support/query overlap: {len(set(episode[:n_sup]) & set(episode[n_sup:]))}")

t0 = time.time()
x, y = train_ds[0]
print(f"Sample image tensor: {tuple(x.shape)}, label: {y}")

train=5778  val=826  test=1656
Training batches per epoch: 115
Episode size: 50
Support labels: [0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4]
Query labels:   [0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4]
Support/query overlap: 0
Sample image tensor: (3, 224, 224), label: 0


In [5]:
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models


class CBAM(nn.Module):
    """Convolutional Block Attention Module (Woo et al., 2018): channel attention, then spatial attention."""

    def __init__(self, channels, reduction=16, kernel_size=7):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels // reduction, channels, 1, bias=False),
        )
        self.spatial = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)
        self.last_spatial_map = None  # kept for attention visualisation

    def forward(self, x):
        channel_att = torch.sigmoid(self.mlp(F.adaptive_avg_pool2d(x, 1)) +
                                    self.mlp(F.adaptive_max_pool2d(x, 1)))
        x = x * channel_att
        pooled = torch.cat([x.mean(dim=1, keepdim=True), x.amax(dim=1, keepdim=True)], dim=1)
        spatial_att = torch.sigmoid(self.spatial(pooled))
        self.last_spatial_map = spatial_att.detach()
        return x * spatial_att


class EmbeddingNet(nn.Module):
    """ImageNet-pretrained CNN backbone, optional CBAM, then global average pooling."""

    def __init__(self, backbone, use_attention, reduction=16):
        super().__init__()
        if backbone == 'resnet18':
            base = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
            self.features = nn.Sequential(*list(base.children())[:-2])
            self.out_dim = 512
        else:
            base = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
            # torchvision's DenseNet applies a ReLU after the dense blocks
            self.features = nn.Sequential(base.features, nn.ReLU(inplace=True))
            self.out_dim = 1024
        self.attention = CBAM(self.out_dim, reduction) if use_attention else None

    def forward(self, x):
        fmap = self.features(x)
        if self.attention is not None:
            fmap = self.attention(fmap)
        return F.adaptive_avg_pool2d(fmap, 1).flatten(1)


class VanillaClassifier(nn.Module):
    """Standard softmax classifier on the same backbone (control model)."""

    def __init__(self, backbone):
        super().__init__()
        self.encoder = EmbeddingNet(backbone, use_attention=False)
        self.head = nn.Linear(self.encoder.out_dim, 5)

    def forward(self, x):
        return self.head(self.encoder(x))


# Re-seed so that newly initialised layers (CBAM, linear head) are identical across reruns
set_seed(SEED)

if MODEL == 'vanilla':
    model = VanillaClassifier(BACKBONE)
    encoder = model.encoder
else:
    model = EmbeddingNet(BACKBONE, CFG['use_attention'], CFG['cbam_reduction'])
    encoder = model
model = model.to(device)

n_params = sum(p.numel() for p in model.parameters())
print(f"Model: {MODEL} | backbone: {BACKBONE} | attention: {CFG['use_attention']} | ordinal loss: {CFG['use_ordinal']}")
print(f"Parameters: {n_params:,}")

model.eval()
with torch.no_grad():
    dummy = torch.randn(2, 3, CFG['img_size'], CFG['img_size'], device=device)
    print(f"Embedding shape: {tuple(encoder(dummy).shape)}")
    if MODEL == 'vanilla':
        print(f"Logit shape: {tuple(model(dummy).shape)}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 115MB/s]


Model: full | backbone: resnet18 | attention: True | ordinal loss: True
Parameters: 11,209,378
Embedding shape: (2, 512)


In [6]:
from sklearn.metrics import (accuracy_score, f1_score, precision_score, recall_score,
                             cohen_kappa_score)

N_CLASSES = 5


def class_prototypes(emb, labels):
    """Mean embedding of each KL grade."""
    return torch.stack([emb[labels == k].mean(dim=0) for k in range(N_CLASSES)])


def ordinal_loss(prototypes, margin):
    """
    Each grade's prototype should lie closer to the adjacent grade than to the grade two steps away:
    mean over k of max(0, d(c_k, c_{k+1}) - d(c_k, c_{k+2}) + margin), k = 0..K-3.
    """
    terms = []
    for k in range(N_CLASSES - 2):
        d_adjacent = torch.norm(prototypes[k] - prototypes[k + 1], p=2)
        d_skip = torch.norm(prototypes[k] - prototypes[k + 2], p=2)
        terms.append(F.relu(d_adjacent - d_skip + margin))
    return torch.stack(terms).mean()


def episode_loss(images, labels):
    """Prototypes come from the support images only; the loss is computed on the query images."""
    n_sup = N_CLASSES * CFG['n_support']
    emb = encoder(images)
    support_emb, query_emb = emb[:n_sup], emb[n_sup:]
    support_lbl, query_lbl = labels[:n_sup], labels[n_sup:]

    prototypes = class_prototypes(support_emb, support_lbl)
    logits = -torch.cdist(query_emb, prototypes)
    ce = F.cross_entropy(logits, query_lbl)

    if CFG['use_ordinal']:
        ord_term = ordinal_loss(prototypes, CFG['ordinal_margin'])
    else:
        ord_term = torch.zeros((), device=images.device)

    loss = ce + CFG['ordinal_alpha'] * ord_term
    return loss, ce.detach(), ord_term.detach(), logits.detach(), query_lbl


@torch.no_grad()
def extract_embeddings(loader):
    encoder.eval()
    embs, lbls = [], []
    for x, y in loader:
        embs.append(encoder(x.to(device, non_blocking=True)).float().cpu())
        lbls.append(y)
    return torch.cat(embs), torch.cat(lbls)


@torch.no_grad()
def predict_protonet(loader, prototypes):
    emb, y = extract_embeddings(loader)
    logits = -torch.cdist(emb, prototypes)
    return dict(labels=y.numpy(), logits=logits.numpy(),
                probs=F.softmax(logits, dim=1).numpy(), preds=logits.argmax(1).numpy(),
                embeddings=emb.numpy(), loss=F.cross_entropy(logits, y).item())


@torch.no_grad()
def predict_vanilla(loader):
    model.eval()
    embs, logits_all, lbls = [], [], []
    for x, y in loader:
        emb = model.encoder(x.to(device, non_blocking=True))
        embs.append(emb.float().cpu())
        logits_all.append(model.head(emb).float().cpu())
        lbls.append(y)
    emb, logits, y = torch.cat(embs), torch.cat(logits_all), torch.cat(lbls)
    return dict(labels=y.numpy(), logits=logits.numpy(),
                probs=F.softmax(logits, dim=1).numpy(), preds=logits.argmax(1).numpy(),
                embeddings=emb.numpy(), loss=F.cross_entropy(logits, y).item())


def compute_metrics(y, pred):
    f1_per_class = f1_score(y, pred, labels=list(range(N_CLASSES)), average=None, zero_division=0)
    metrics = dict(
        accuracy=accuracy_score(y, pred),
        f1_macro=f1_score(y, pred, average='macro', zero_division=0),
        precision_macro=precision_score(y, pred, average='macro', zero_division=0),
        recall_macro=recall_score(y, pred, average='macro', zero_division=0),
        mae=np.abs(y - pred).mean(),
        qwk=cohen_kappa_score(y, pred, weights='quadratic'),
        within_one=(np.abs(y - pred) <= 1).mean(),
    )
    for k in range(N_CLASSES):
        metrics[f'f1_kl{k}'] = f1_per_class[k]
    return {k: float(v) for k, v in metrics.items()}


def evaluate(loader, train_emb=None):
    """
    Vanilla: softmax over the classifier logits.
    Prototypical models: class prototypes are the mean embeddings of the TRAINING images of each
    grade; labels of the evaluated split are used only to compute metrics, never to build prototypes.
    """
    if MODEL == 'vanilla':
        out = predict_vanilla(loader)
    else:
        if train_emb is None:
            train_emb = extract_embeddings(train_eval_loader)
        out = predict_protonet(loader, class_prototypes(*train_emb))
    out['metrics'] = compute_metrics(out['labels'], out['preds'])
    return out


# Quick check of the loss and the evaluation pipeline with the ImageNet weights (before training).
if MODEL != 'vanilla':
    episode = next(iter(EpisodeSampler(train_ds.labels, CFG['n_support'], CFG['n_query'], 1, seed=999)))
    xb = torch.stack([train_eval_ds[i][0] for i in episode]).to(device)
    yb = torch.as_tensor(train_ds.labels[episode]).to(device)
    encoder.eval()
    with torch.no_grad():
        loss, ce, ord_term, _, _ = episode_loss(xb, yb)
    print(f"Episode loss: {loss.item():.4f} (CE {ce.item():.4f}, ordinal {ord_term.item():.4f})")

t0 = time.time()
val_check = evaluate(val_loader)
m = val_check['metrics']
print(f"Validation before training: acc={m['accuracy']:.4f}  F1-macro={m['f1_macro']:.4f}  "
      f"({time.time() - t0:.0f} s)")

Episode loss: 1.8861 (CE 1.8676, ordinal 0.0370)
Validation before training: acc=0.3232  F1-macro=0.2888  (20 s)


In [7]:
set_seed(SEED)

optimizer = torch.optim.Adam(model.parameters(), lr=CFG['lr'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=CFG['lr_factor'], patience=CFG['lr_patience'])

BEST_PATH = os.path.join(RUN_DIR, 'best_model.pth')
LAST_PATH = os.path.join(RUN_DIR, 'last_checkpoint.pth')
HISTORY_PATH = os.path.join(RUN_DIR, 'history.json')


def get_rng_state():
    state = dict(python=random.getstate(), numpy=np.random.get_state(),
                 torch=torch.get_rng_state(), cuda=torch.cuda.get_rng_state_all(),
                 loader=loader_gen.get_state())
    if MODEL != 'vanilla':
        state['sampler'] = train_sampler.rng.bit_generator.state
    return state


def set_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'])
    torch.cuda.set_rng_state_all(state['cuda'])
    loader_gen.set_state(state['loader'])
    if MODEL != 'vanilla':
        train_sampler.rng.bit_generator.state = state['sampler']


# Resume if a previous session of this run was interrupted
history, best_f1, best_epoch, start_epoch = [], -1.0, 0, 1
if os.path.exists(LAST_PATH):
    ckpt = torch.load(LAST_PATH, map_location='cpu', weights_only=False)
    model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    history, best_f1, best_epoch = ckpt['history'], ckpt['best_f1'], ckpt['best_epoch']
    start_epoch = ckpt['epoch'] + 1
    set_rng_state(ckpt['rng'])
    print(f"Resuming {RUN_NAME} from epoch {start_epoch}")


def train_one_epoch():
    model.train()
    sum_loss = sum_ce = sum_ord = 0.0
    correct = total = 0
    for x, y in train_loader:
        x = x.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)
        if MODEL == 'vanilla':
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            ce, ord_term, target = loss.detach(), torch.zeros(()), y
            logits = logits.detach()
        else:
            loss, ce, ord_term, logits, target = episode_loss(x, y)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

        sum_loss += loss.item()
        sum_ce += ce.item()
        sum_ord += float(ord_term)
        correct += (logits.argmax(1) == target).sum().item()
        total += target.numel()

    n = len(train_loader)
    return dict(train_loss=sum_loss / n, train_ce=sum_ce / n,
                train_ord=sum_ord / n, train_acc=correct / total)


print(f"Training {RUN_NAME}, epochs {start_epoch}-{CFG['epochs']}")
for epoch in range(start_epoch, CFG['epochs'] + 1):
    t0 = time.time()
    lr = optimizer.param_groups[0]['lr']

    train_stats = train_one_epoch()
    val_out = evaluate(val_loader)
    vm = val_out['metrics']
    scheduler.step(val_out['loss'])

    improved = vm['f1_macro'] > best_f1
    if improved:
        best_f1, best_epoch = vm['f1_macro'], epoch
        torch.save(model.state_dict(), BEST_PATH)

    history.append(dict(epoch=epoch, lr=lr, **train_stats, val_loss=val_out['loss'],
                        **{f'val_{k}': v for k, v in vm.items()},
                        seconds=time.time() - t0))
    with open(HISTORY_PATH, 'w') as f:
        json.dump(history, f, indent=1)

    # Write to a temporary file first so a disconnect cannot leave a corrupted checkpoint
    torch.save(dict(epoch=epoch, model=model.state_dict(), optimizer=optimizer.state_dict(),
                    scheduler=scheduler.state_dict(), history=history, best_f1=best_f1,
                    best_epoch=best_epoch, rng=get_rng_state()), LAST_PATH + '.tmp')
    os.replace(LAST_PATH + '.tmp', LAST_PATH)

    print(f"Epoch {epoch:2d}/{CFG['epochs']} | lr {lr:.1e} | "
          f"train loss {train_stats['train_loss']:.4f} acc {train_stats['train_acc']:.4f} | "
          f"val loss {val_out['loss']:.4f} acc {vm['accuracy']:.4f} F1 {vm['f1_macro']:.4f} "
          f"KL1 {vm['f1_kl1']:.3f} KL4 {vm['f1_kl4']:.3f} QWK {vm['qwk']:.3f} | "
          f"{time.time() - t0:.0f} s{'  *' if improved else ''}")

print(f"\nBest validation F1-macro: {best_f1:.4f} (epoch {best_epoch})")

Training full_resnet18_seed0, epochs 1-30
Epoch  1/30 | lr 1.0e-04 | train loss 1.3441 acc 0.4223 | val loss 1.1349 acc 0.4806 F1 0.5019 KL1 0.226 KL4 0.733 QWK 0.628 | 44 s  *
Epoch  2/30 | lr 1.0e-04 | train loss 0.9660 acc 0.5614 | val loss 1.0512 acc 0.5194 F1 0.5609 KL1 0.269 KL4 0.818 QWK 0.682 | 42 s  *
Epoch  3/30 | lr 1.0e-04 | train loss 0.8994 acc 0.6049 | val loss 0.9778 acc 0.5630 F1 0.6100 KL1 0.272 KL4 0.867 QWK 0.707 | 46 s  *
Epoch  4/30 | lr 1.0e-04 | train loss 0.8215 acc 0.6344 | val loss 0.9539 acc 0.5847 F1 0.6162 KL1 0.323 KL4 0.815 QWK 0.729 | 45 s  *
Epoch  5/30 | lr 1.0e-04 | train loss 0.7870 acc 0.6431 | val loss 0.9726 acc 0.5593 F1 0.6002 KL1 0.269 KL4 0.852 QWK 0.740 | 45 s
Epoch  6/30 | lr 1.0e-04 | train loss 0.7683 acc 0.6539 | val loss 0.9449 acc 0.5860 F1 0.6281 KL1 0.348 KL4 0.842 QWK 0.737 | 44 s  *
Epoch  7/30 | lr 1.0e-04 | train loss 0.7506 acc 0.6623 | val loss 0.9175 acc 0.5714 F1 0.6140 KL1 0.292 KL4 0.871 QWK 0.768 | 45 s
Epoch  8/30 | lr 1.

In [8]:
from datetime import datetime, timezone
from sklearn.metrics import confusion_matrix

BEST_PATH = os.path.join(RUN_DIR, 'best_model.pth')
LAST_PATH = os.path.join(RUN_DIR, 'last_checkpoint.pth')
HISTORY_PATH = os.path.join(RUN_DIR, 'history.json')

with open(HISTORY_PATH) as f:
    history = json.load(f)
if len(history) != CFG['epochs']:
    raise RuntimeError(f"Training incomplete: {len(history)}/{CFG['epochs']} epochs. Run the training cell first.")
best = max(history, key=lambda h: h['val_f1_macro'])

model.load_state_dict(torch.load(BEST_PATH, map_location=device))

# The test set is evaluated once, with the checkpoint selected on the validation set
train_emb = extract_embeddings(train_eval_loader)
val_out = evaluate(val_loader, train_emb)
test_out = evaluate(test_loader, train_emb)
cm = confusion_matrix(test_out['labels'], test_out['preds'], labels=list(range(N_CLASSES)))

arrays = dict(
    train_emb=train_emb[0].numpy().astype(np.float16),
    train_labels=train_emb[1].numpy(),
    train_files=np.array(train_eval_ds.files),
    val_files=np.array(val_ds.files),
    test_files=np.array(test_ds.files),
)
if MODEL != 'vanilla':
    arrays['prototypes'] = class_prototypes(*train_emb).numpy()
for split, out in [('val', val_out), ('test', test_out)]:
    arrays[f'{split}_emb'] = out['embeddings'].astype(np.float16)
    arrays[f'{split}_labels'] = out['labels']
    arrays[f'{split}_logits'] = out['logits'].astype(np.float32)
    arrays[f'{split}_probs'] = out['probs'].astype(np.float32)
    arrays[f'{split}_preds'] = out['preds']
np.savez_compressed(os.path.join(RUN_DIR, 'outputs.npz'), **arrays)

results = dict(
    run_name=RUN_NAME,
    config=CFG,
    best_epoch=best['epoch'],
    best_val_f1_macro=best['val_f1_macro'],
    train_minutes=sum(h['seconds'] for h in history) / 60,
    val=val_out['metrics'],
    test=test_out['metrics'],
    test_confusion_matrix=cm.tolist(),
    finished_utc=datetime.now(timezone.utc).isoformat(timespec='seconds'),
)
with open(os.path.join(RUN_DIR, 'results.json'), 'w') as f:
    json.dump(results, f, indent=2)

# The resume checkpoint is no longer needed once results are saved
if os.path.exists(LAST_PATH):
    os.remove(LAST_PATH)

tm = test_out['metrics']
print(f"Run: {RUN_NAME} | best epoch {best['epoch']} (val F1-macro {best['val_f1_macro']:.4f})")
print(f"Test accuracy   {tm['accuracy']:.4f}")
print(f"Test F1-macro   {tm['f1_macro']:.4f}")
print("Test F1 per KL  " + "  ".join(f"KL{k} {tm[f'f1_kl{k}']:.4f}" for k in range(N_CLASSES)))
print(f"Test MAE        {tm['mae']:.4f}")
print(f"Test QWK        {tm['qwk']:.4f}")
print(f"Within 1 grade  {tm['within_one']:.4f}")
print("\nTest confusion matrix (rows = true KL 0-4, columns = predicted):")
print(cm)
print("\nSaved files:")
for fname in sorted(os.listdir(RUN_DIR)):
    size_mb = os.path.getsize(os.path.join(RUN_DIR, fname)) / 1e6
    print(f"  {fname:20s} {size_mb:7.1f} MB")

Run: full_resnet18_seed0 | best epoch 29 (val F1-macro 0.6964)
Test accuracy   0.6479
Test F1-macro   0.6753
Test F1 per KL  KL0 0.7442  KL1 0.3764  KL2 0.6441  KL3 0.7751  KL4 0.8367
Test MAE        0.3907
Test QWK        0.8291
Within 1 grade  0.9614

Test confusion matrix (rows = true KL 0-4, columns = predicted):
[[451 163  25   0   0]
 [ 91 131  69   5   0]
 [ 31 103 276  37   0]
 [  0   3  40 174   6]
 [  0   0   0  10  41]]

Saved files:
  best_model.pth          44.9 MB
  config.json              0.0 MB
  history.json             0.0 MB
  outputs.npz              7.9 MB
  results.json             0.0 MB
